# Full Benchmark: IMDB Sentiment Classification

This notebook runs the complete benchmark across multiple embeddings and models.

**Experiments included:**
- TF-IDF × (LogReg, RandomForest, AdaBoost, LSTM)
- Word2Vec CBOW × (LogReg, RandomForest, AdaBoost, LSTM)
- Word2Vec Skip-gram × (LogReg, RandomForest, AdaBoost, LSTM)
- BERT variants × (LogReg, LSTM)

**Time required:** Several hours (depending on sample size and GPU availability)

**Recommendations:**
- Start with small sample (e.g., 1000) to test pipeline
- Use GPU runtime for BERT experiments
- Save results to Google Drive for persistence

## Setup

In [ ]:
# Clone repository if in Colab
import os

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('Embedding_models_with_Classification'):
        !git clone https://github.com/rylex27-z/Embedding_models_with_Classification.git
    %cd Embedding_models_with_Classification

# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
# Data path - UPDATE THIS
DATA_PATH = '/content/aclImdb.zip'  # or '/content/drive/MyDrive/data/aclImdb.zip'

# Sample size (set to None for full dataset)
SAMPLE_SIZE = 1000  # Start with small sample; set to None for full run

# CV configuration (reduce for faster testing)
CV_CONFIG = {
    'n_splits': 5,      # Number of folds
    'n_repeats': 4,     # Number of repeats (total = n_splits × n_repeats)
    'random_seeds': [42, 123, 456, 789]
}

# Which experiments to run
RUN_TFIDF = True
RUN_WORD2VEC = True
RUN_BERT = False  # Set to True if you have GPU and time

# BERT models to test (if RUN_BERT=True)
BERT_MODELS = [
    'distilbert-base-uncased',  # Fastest
    'bert-base-uncased',
    # 'roberta-base',
    # 'albert-base-v2',
]

## Load Data

In [ ]:
import sys
sys.path.insert(0, 'src')

from src.data_loader import load_and_preprocess_imdb
from src.embeddings import get_embedding
from src.models import get_model
from src.evaluation import run_full_evaluation
from src.utils import save_results_csv, get_environment_info
import numpy as np
import pandas as pd
from datetime import datetime

print("Loading IMDB dataset...")
train_texts, train_labels, test_texts, test_labels = load_and_preprocess_imdb(
    zip_path=DATA_PATH,
    sample_size=SAMPLE_SIZE,
    preprocess=True,
    random_seed=42
)

train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

print(f"\nLoaded {len(train_texts)} train and {len(test_texts)} test samples")

## Helper Function

In [ ]:
# Store all results
all_results = []

def run_and_save_experiment(embedding_type, embedding_variant, model_type, embedding_config=None, model_config=None):
    """Run single experiment and save results."""
    print(f"\n{'='*80}")
    print(f"Running: {embedding_type} ({embedding_variant}) + {model_type}")
    print(f"{'='*80}")
    
    try:
        # Create embedding
        embedding = get_embedding(embedding_type, embedding_variant, embedding_config or {})
        
        # Create model
        if model_type == 'lstm':
            # Get input dimension
            temp_emb = embedding.fit_transform([train_texts[0]])
            input_dim = temp_emb.shape[1]
            model = get_model(model_type, input_dim=input_dim, config=model_config or {})
        else:
            model = get_model(model_type, config=model_config or {})
        
        # Run evaluation
        results = run_full_evaluation(
            embedding_obj=embedding,
            model_obj=model,
            X_train=train_texts,
            y_train=train_labels,
            X_test=test_texts,
            y_test=test_labels,
            embedding_type=embedding_type,
            embedding_variant=embedding_variant,
            model_type=model_type,
            cv_config=CV_CONFIG,
            verbose=True
        )
        
        # Save to list
        all_results.append({
            'embedding_type': embedding_type,
            'embedding_variant': embedding_variant,
            'model_type': model_type,
            'results': results
        })
        
        print(f"✓ Completed successfully")
        return results
        
    except Exception as e:
        print(f"✗ Error: {e}")
        import traceback
        traceback.print_exc()
        return None

## Run Experiments

### TF-IDF Experiments

In [ ]:
if RUN_TFIDF:
    tfidf_config = {'max_features': 5000, 'ngram_range': (1, 2)}
    
    for model in ['logreg', 'randomforest', 'adaboost', 'lstm']:
        run_and_save_experiment('tfidf', 'max_features_5000', model, 
                              embedding_config=tfidf_config)

### Word2Vec Experiments

In [ ]:
if RUN_WORD2VEC:
    # CBOW
    w2v_cbow_config = {'vector_size': 100, 'window': 5, 'sg': 0, 'epochs': 10}
    for model in ['logreg', 'randomforest', 'adaboost', 'lstm']:
        run_and_save_experiment('word2vec', 'cbow', model, 
                              embedding_config=w2v_cbow_config)
    
    # Skip-gram
    w2v_sg_config = {'vector_size': 100, 'window': 5, 'sg': 1, 'epochs': 10}
    for model in ['logreg', 'randomforest', 'adaboost', 'lstm']:
        run_and_save_experiment('word2vec', 'skipgram', model, 
                              embedding_config=w2v_sg_config)

### BERT Experiments (requires GPU)

In [ ]:
if RUN_BERT:
    bert_config = {'max_length': 256, 'batch_size': 8, 'pooling': 'cls'}
    
    for bert_model in BERT_MODELS:
        bert_config['model_name'] = bert_model
        
        # Usually just LogReg with BERT (LSTM would be redundant)
        for model in ['logreg']:
            run_and_save_experiment('bert', bert_model, model, 
                                  embedding_config=bert_config)

## Aggregate and Save Results

In [ ]:
# Aggregate CV results (long format)
cv_long_records = []
summary_records = []

for exp in all_results:
    if exp['results'] is not None:
        # CV long
        if 'cv_results' in exp['results']:
            cv_df = exp['results']['cv_results'].copy()
            cv_df['embedding_type'] = exp['embedding_type']
            cv_df['embedding_variant'] = exp['embedding_variant']
            cv_df['model_type'] = exp['model_type']
            cv_long_records.append(cv_df)
        
        # Summary
        summary = {
            'embedding_type': exp['embedding_type'],
            'embedding_variant': exp['embedding_variant'],
            'model_type': exp['model_type']
        }
        if 'cv_summary' in exp['results']:
            summary.update(exp['results']['cv_summary'])
        if 'test_metrics' in exp['results']:
            for k, v in exp['results']['test_metrics'].items():
                summary[f'test_{k}'] = v
        summary_records.append(summary)

# Create DataFrames
if cv_long_records:
    cv_long_df = pd.concat(cv_long_records, ignore_index=True)
else:
    cv_long_df = pd.DataFrame()

summary_df = pd.DataFrame(summary_records)

# Save
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if not cv_long_df.empty:
    cv_long_df.to_csv(f'reports/results/results_long_{timestamp}.csv', index=False)
    print(f"Saved: reports/results/results_long_{timestamp}.csv")

summary_df.to_csv(f'reports/results/results_summary_{timestamp}.csv', index=False)
print(f"Saved: reports/results/results_summary_{timestamp}.csv")

# Display summary
print("\n" + "="*80)
print("SUMMARY RESULTS")
print("="*80)
print(summary_df[['embedding_type', 'embedding_variant', 'model_type', 
                  'accuracy_mean', 'f1_mean', 'runtime_mean']].to_string(index=False))

## Generate Markdown Table

In [ ]:
from src.utils import format_results_table

md_table = format_results_table(summary_df)

# Save markdown
with open(f'reports/results/results_table_{timestamp}.md', 'w') as f:
    f.write("# IMDB Sentiment Classification Results\n\n")
    f.write(md_table)

print(md_table)

## Commit Results to GitHub (Optional)

If you want to save results back to the repository:

In [ ]:
# Configure git
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"

# Add and commit results
# !git add reports/results/*.csv reports/results/*.md
# !git commit -m "Add experiment results from Colab"
# !git push

## Next Steps

1. Copy result files to Google Drive for safekeeping
2. Update `reports/final_report.md` with your findings
3. Run additional experiments with different hyperparameters
4. Analyze errors and patterns in misclassifications

Happy benchmarking! 📊